In [127]:
import numpy as np
from scipy.stats import norm
from matplotlib import pyplot as plt

In [10]:
data_size = 1000
# generate random numbers from N(0,1)
normal = norm.rvs(size=data_size,loc=0,scale=1)
anormal_s = norm.rvs(size=data_size,loc=0,scale=0.1)
anormal_b = norm.rvs(size=data_size,loc=0,scale=10)

x = np.array((normal, anormal_s, anormal_b))
x = x.reshape((data_size*3))

mixed = x

ones = np.ones(data_size)
x = np.array((normal, 'n'*data_size, 
              anormal_s, 's'*data_size, 
              anormal_b, 'b'*data_size), dtype=object)

x = np.array((normal, ones, 
              anormal_s, ones*2, 
              anormal_b, ones*3))
print(x)
#x.reshape((data_size*3, data_size*3))

[[ 1.85734222e+00  1.58546182e+00 -2.57726192e-01 ... -1.18682074e+00
   6.32107388e-01 -2.03309712e+00]
 [ 1.00000000e+00  1.00000000e+00  1.00000000e+00 ...  1.00000000e+00
   1.00000000e+00  1.00000000e+00]
 [ 8.32703989e-02  5.44686635e-03 -8.38249145e-02 ... -2.60626126e-02
   3.50923851e-02  5.23437381e-02]
 [ 2.00000000e+00  2.00000000e+00  2.00000000e+00 ...  2.00000000e+00
   2.00000000e+00  2.00000000e+00]
 [ 1.04171085e+01 -1.58101541e+00  8.37267571e+00 ...  1.35620475e+01
   4.69726898e+00 -6.67240947e+00]
 [ 3.00000000e+00  3.00000000e+00  3.00000000e+00 ...  3.00000000e+00
   3.00000000e+00  3.00000000e+00]]


In [11]:
def compute_entropy(a):
    pdfs = norm.pdf(a)
    h = - pdfs/len(a) @ np.log(pdfs)
    #h_norm = h/len(a)
    return h

#compute_entropy(x)

In [12]:
gauss_entro = compute_entropy(norm.rvs(size=1000000,loc=0,scale=1))
print(gauss_entro)

0.3298737310422126


In [14]:
def print_name(name, var):
    print(name + ": \t" + str(var))

In [16]:
print_name("Norm Entro", 0.5 * np.log2(2*np.pi * np.e))

Norm Entro: 	2.047095585180641


In [17]:
np.random.shuffle(mixed)
print_name("normal", compute_entropy(normal))
print_name("anormal small", compute_entropy(anormal_s))
print_name("anormal big", compute_entropy(anormal_b))
print_name("mixed", compute_entropy(mixed))

normal: 	0.3286686694146233
anormal small: 	0.36675285957037385
anormal big: 	0.05327445693385445
mixed: 	0.24956532863961722


In [18]:
def stupid_rotation(A, leave_out):
    dummy = np.copy(A)[1:]
    if leave_out != 0:
        dummy[leave_out-1] = A[0]
    return dummy

In [19]:
def find_worst_data_point(A):
    worst_entro_diff = 0
    worst_idx = -1
    
    for i in range(len(A)):
        dummy = stupid_rotation(A, i)
        entro = compute_entropy(dummy)
        
        entro_diff = entro - gauss_entro
        #print(entro_diff)
        if np.abs(entro_diff) > worst_entro_diff:
            worst_entro_diff = entro_diff
            worst_idx = i
    return worst_idx
        
find_worst_data_point(anormal_s)

195

In [67]:
def KL_div(A):
    mean = A.mean()
    var = A.var()
    
    norm_pdf = norm.pdf(A)
    calc_pdf = norm.pdf(A, mean, var)
    
    return norm_pdf @ np.log(norm_pdf / calc_pdf)
    
print_name("normal", KL_div(normal))

normal: 	3.423707283156987


In [130]:
lst = []
for _ in range(10000):
    #lst.append(print_name("Norm KL", KL_div(norm.rvs(size=1000))))
    lst.append(KL_div(norm.rvs(size=1000)))

In [132]:
#plt.hist(lst)

In [134]:
print_name("normal", KL_div(normal))
print_name("anormal small", KL_div(anormal_s))
print_name("anormal big", KL_div(anormal_b))
print_name("mixed", KL_div(mixed[:data_size]))

normal: 	3.423707283156987
anormal small: 	17435.152617464173
anormal big: 	146.91221807908852
mixed: 	810.7686581896116
